# Seen and Unseen — Experiment 2 TRAINING (locked; run only after registration, the freeze and the tag)

Trains the 110 registered Experiment 2 runs (130 under the B fallback) from the frozen git tag, with the owner's pipeline at its registered commit and the signed freeze manifest in `approvals/exp2/`. Run files go to Google Drive, so a restarted session resumes.

**Sealed outputs.** Each run file holds per-episode judge scores, which are outcome data. They stay on Drive in `SEALED_registered_runs/`. Nobody opens or downloads them, and nobody runs the analyzer here: the statistics reviewer analyzes them after the December 8 amendment (preregistration v3.4, section A). The log shows run status and time only.

**Refusals built in:** the runner will not start without a signed freeze manifest that matches the config, selection, support and code; without the owner's repository at the frozen commit with an unedited `experiment/wm.py`; on another hardware class or runtime unless `DEVIATION` gives a reason (logged); or if the judge fails its validity gate.

**Runtime:** about 2–3 h on one A100. After a disconnect: Runtime → Run all; finished runs are skipped.

In [ ]:
#@title 0. Configuration
FREEZE_TAG = "exp2-freeze-YYYYMMDD"   #@param {type:"string"}   # the dated git tag the paper lead created at the freeze
EXP2_DRIVE_DIR = "/content/drive/MyDrive/seen-and-unseen/exp2"  #@param {type:"string"}
DEVIATION = ""                        #@param {type:"string"}   # only if the GPU or runtime differs from the freeze; the reason is logged
RETRY_FAILED = False                  #@param {type:"boolean"}  # True once, to rerun infrastructure failures with the same seed
OWNER_COMMIT = "ed416e15ab9e4fd7c7b9ba4c51bdb343b4a28ce5"
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch to a GPU runtime)")
from google.colab import drive
drive.mount("/content/drive")
CACHE = f"{EXP2_DRIVE_DIR}/kuka"
assert os.path.isdir(CACHE), f"the Kuka cache is missing at {CACHE}"
print("cache shards:", len([f for f in os.listdir(CACHE) if f.endswith(".npz")]))
RUNS = f"{EXP2_DRIVE_DIR}/SEALED_registered_runs"
os.makedirs(RUNS, exist_ok=True)
print("run files already written:", len([f for f in os.listdir(RUNS) if f.endswith(".json") and "_s" in f]))

In [ ]:
assert "FREEZE_TAG" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 1. Code and the signed freeze at the frozen tag
%cd /content
import os, json
if not os.path.exists("trajectory-lineage"):
    !git clone -q https://github.com/physicalreasoning/trajectory-lineage.git
%cd /content/trajectory-lineage
!git fetch -q --tags && git checkout -q {FREEZE_TAG} && git rev-parse HEAD
FROZEN = "/content/trajectory-lineage/approvals/exp2"
FREEZE = f"{FROZEN}/exp2_freeze_manifest.json"
frozen = json.load(open(FREEZE))
print("freeze status:", frozen["status"], "| hardware class:", frozen["environment"]["hardware_class"], "| owner commit:", frozen["owner_pipeline"]["commit"][:12])
assert frozen["status"] == "signed", "the freeze manifest at this tag is not signed; do not train"
assert frozen["owner_pipeline"]["commit"] == OWNER_COMMIT, "the freeze names another owner commit than this notebook"
pins = [f"{k}=={v}" for k, v in frozen["environment"]["runtime"].items() if k in ("numpy", "scipy", "scikit-learn", "pandas", "transformers") and v != "absent"]
PIN_ARGS = " ".join(pins)
print("installing the frozen versions:", PIN_ARGS)
!pip install -q {PIN_ARGS}

In [ ]:
assert "FREEZE_TAG" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 2. The owner's pipeline at the registered commit (wm.py unedited) and the cache
%cd /content
import os
if not os.path.exists("rwmo"):
    !git clone -q https://github.com/subhashjprasad/robot-world-model-optimism-research.git rwmo
!git -C /content/rwmo fetch -q && git -C /content/rwmo checkout -q -f {OWNER_COMMIT} && git -C /content/rwmo rev-parse HEAD
!mkdir -p /content/rwmo/data && rm -rf /content/rwmo/data/kuka && ln -s {CACHE} /content/rwmo/data/kuka
!ls /content/rwmo/data/kuka | wc -l

In [ ]:
assert "FREEZE_TAG" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 3. Train the registered runs (resumable; blinded log)
%cd /content/trajectory-lineage/exp2-kuka-matched
EXTRA = (f'--deviation "{DEVIATION}"' if DEVIATION else "") + (" --retry-failed" if RETRY_FAILED else "")
!PYTHONPATH=. python -m registered.run_registered --owner-repo /content/rwmo --npz-dir /content/rwmo/data/kuka --manifest {FROZEN}/manifest.jsonl --selection {FROZEN}/selection.json --support {FROZEN}/support.json --freeze {FREEZE} --out {RUNS} {EXTRA} 2>&1 | grep -v -i "warn" | tail -n 150

In [ ]:
assert "FREEZE_TAG" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 4. Blinded accounting (status counts only; no scores)
import json, glob
expected = json.load(open(f"{FROZEN}/selection.json"))["n_runs"]
status = {}
for path in glob.glob(f"{RUNS}/*_s*.json"):
    s = json.load(open(path)).get("status")
    status[s] = status.get(s, 0) + 1
print(f"run files: {sum(status.values())} of {expected}; by status: {status}")
print("If any run shows infrastructure_failure: set RETRY_FAILED = True and run cells 0-3 once more.")

### After all runs complete
Do not open, copy or download `SEALED_registered_runs/`. Tell the paper lead that training finished, with the counts from cell 4. The statistics reviewer runs `registered.analyze` on these files after the December 8 amendment.